In [ ]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_rows", None)

HIST_ROOTS = {
    "ehrshot": Path("/storage/shared/ehr-shot/filtered_labs/eic/compute_benchmark/histogram_grid/profiling/ehrshot"),
    "mimic": Path("/storage/shared/mimic-iv/meds_v0.4.0_mimicv3.1/eic/compute_benchmark/histogram_grid/profiling/mimic"),
}

BASELINE_ROOTS = {
    "ehrshot": Path("/storage/shared/ehr-shot/filtered_labs/eic/compute_benchmark/baseline/profiling/ehrshot"),
    "mimic": Path("/storage/shared/mimic-iv/meds_v0.4.0_mimicv3.1/eic/compute_benchmark/baseline/profiling/mimic"),
}

DISPLAY_NAMES = {
    "ehrshot": "EHRSHOT",
    "mimic": "MIMIC",
}


def load_summaries(root: Path) -> pd.DataFrame:
    rows = []
    for summary_path in root.rglob("*_summary.json"):
        with open(summary_path) as f:
            payload = json.load(f)
        rows.append(payload | {"_path": str(summary_path)})
    return pd.DataFrame(rows)


def fmt_median_min_max(series: pd.Series) -> str:
    s = series.dropna()
    return f"{s.median():.1f} [{s.min():.1f}-{s.max():.1f}]"


In [ ]:
def compute_training_cost_row(dataset: str) -> pd.DataFrame:
    dataset = dataset.lower()
    hist_df = load_summaries(HIST_ROOTS[dataset])
    base_df = load_summaries(BASELINE_ROOTS[dataset])

    hist_df = hist_df[hist_df["exit_code"] == 0].copy()
    base_df = base_df[base_df["exit_code"] == 0].copy()

    hist_df["K"] = pd.to_numeric(hist_df["_path"].str.extract(r"_K(\d+)_", expand=False))
    hist_df["resolution"] = hist_df["_path"].str.extract(
        r"/(day|week|month|quarter_annual|semi_annual)/", expand=False
    )
    hist_df["wall_time_min"] = hist_df["wall_time_seconds"] / 60.0

    train_stages = hist_df[hist_df["stage"].isin(["fit_em", "train_histogram_ar"])]
    expert_costs = (
        train_stages
        .pivot_table(
            index=["dataset", "resolution", "K"],
            columns="stage",
            values="wall_time_min",
            aggfunc="first",
        )
        .reset_index()
    )
    expert_costs["coarse_expert_train_time_min"] = (
        expert_costs["fit_em"] + expert_costs["train_histogram_ar"]
    )

    baseline_train = base_df.loc[
        base_df["stage"] == "baseline_pretrain", "wall_time_seconds"
    ].iloc[0] / 60.0
    expert_times = expert_costs["coarse_expert_train_time_min"].dropna()

    return pd.DataFrame([{
        "Dataset": DISPLAY_NAMES[dataset],
        "Baseline train time (min)": round(baseline_train, 1),
        "fit_em (median [min-max])": fmt_median_min_max(expert_costs["fit_em"]),
        "train_histogram_ar (median [min-max])": fmt_median_min_max(expert_costs["train_histogram_ar"]),
        "Coarse expert train time (median [min-max])": fmt_median_min_max(expert_times),
        "# coarse experts (R)": int(len(expert_times)),
        "Average added cost per expert (min)": round(expert_times.mean(), 1),
        "Total MoRGen training cost (min)": round(baseline_train + expert_times.mean(), 1),
    }])


def compute_training_cost_table(datasets=("ehrshot", "mimic")) -> pd.DataFrame:
    return pd.concat([compute_training_cost_row(ds) for ds in datasets], ignore_index=True)


compute_training_cost_table()


In [ ]:
def compute_inference_latency_table(datasets, num_patients):
    rows = []

    for dataset in datasets:
        hist_df = load_summaries(HIST_ROOTS[dataset])
        base_df = load_summaries(BASELINE_ROOTS[dataset])

        hist_df = hist_df[(hist_df["exit_code"] == 0) & (hist_df["stage"] == "histogram_inference")].copy()
        base_df = base_df[(base_df["exit_code"] == 0) & (base_df["stage"] == "baseline_inference")].copy()

        n_patients = num_patients[dataset]
        baseline_pp_1 = float(base_df["wall_time_seconds"].iloc[0]) / n_patients
        morgen_seq_pp_1 = float(hist_df["wall_time_seconds"].sum()) / n_patients
        morgen_par_pp_1 = float(hist_df["wall_time_seconds"].max()) / n_patients

        for method, per_patient_1 in [
            ("Baseline", baseline_pp_1),
            ("MoRGen (sequential)", morgen_seq_pp_1),
            ("MoRGen (parallel)", morgen_par_pp_1),
        ]:
            rows.append({
                "Dataset": DISPLAY_NAMES[dataset],
                "Method": method,
                "Per-patient latency, 1 trajectory (ms)": round(per_patient_1 * 1000, 2),
                "Relative to baseline": f"{per_patient_1 / baseline_pp_1:.1f}x",
            })

    return pd.DataFrame(rows)


compute_inference_latency_table(
    ["ehrshot", "mimic"],
    {"ehrshot": 4_493, "mimic": 20_232},
)
